# NLP Assignment 3
# Chatbot using Hugging Face Transformers

In [ ]:
# Cell 1: Load Model and Tokenizer

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print(f"GPU available: {torch.cuda.is_available()}")

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if torch.cuda.is_available():
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    device = "cuda"
else:
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32
    )
    device = "cpu"
    model.to(device)

model.eval()

print("✅ TinyLlama chatbot model loaded successfully!")

In [ ]:
# Cell 2: Chatbot

def get_response(user_input, chat_history):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful, intelligent, and friendly AI assistant. "
                "Answer factual questions clearly and correctly in simple language. "
                "For casual conversation, be warm and natural."
            )
        }
    ] + chat_history + [
        {"role": "user", "content": user_input}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=True,
            temperature=0.6,
            top_k=40,
            top_p=0.9,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = output[:, inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(
        new_tokens[0],
        skip_special_tokens=True
    ).strip()

    return response


print("Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.\n")

chat_history = []

while True:
    user_input = input("You: ").strip()

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a great day!")
        break

    if user_input == "":
        print("Chatbot: Please type something.\n")
        continue

    response = get_response(user_input, chat_history)

    print("Chatbot:", response, "\n")

    chat_history.append({"role": "user", "content": user_input})
    chat_history.append({"role": "assistant", "content": response})

    # Keep only the last 3 exchanges
    if len(chat_history) > 6:
        chat_history = chat_history[-6:]

# TinyLlama-Based AI Chatbot

This chatbot uses the TinyLlama-1.1B-Chat model from Hugging Face. Unlike basic conversational models such as DialoGPT, TinyLlama is instruction-tuned, which means it can handle both casual conversation and factual question answering using a single model.

The first code cell loads the tokenizer and model. If a GPU is available, the model is loaded in float16 mode to reduce memory usage and improve speed. Otherwise, it runs on the CPU using float32.

The second code cell defines the chatbot logic:

- A system prompt is added to guide the chatbot’s behavior. It tells the model to be helpful, friendly, and accurate.
- The user's previous messages and the chatbot's replies are stored in `chat_history`, allowing the model to remember the conversation context.
- The `apply_chat_template()` function converts the conversation into the correct format expected by TinyLlama.
- The `generate()` function produces the chatbot’s response using controlled sampling settings such as:
  - `temperature=0.6` to keep answers balanced and less random
  - `top_k=40` and `top_p=0.9` to improve response quality
  - `repetition_penalty=1.15` and `no_repeat_ngram_size=3` to reduce repeated words or phrases

To avoid exceeding the token limit, only the last three conversation exchanges are kept in memory.

This chatbot can answer general questions such as “What is NLP?” as well as respond naturally to casual messages like “Hi” or “How are you?”.